# 02 — Nettoyage des données (jury data scientist)

**Cadre projet :** préparer une cohorte d’entraînement **sans fuite** pour prédire le risque de réadmission 30 j (sorties à domicile uniquement — décès exclus).

**Objectif :** charger les CSV, vérifier les colonnes, pseudonymiser l’identité, appliquer les règles qualité, construire la cohorte domicile + split patient-level.

> Données **synthétiques** — exercice pédagogique, jamais une décision clinique réelle.

**Avant :** [Cadre](../00_guide/01_cadre_projet.ipynb) · [Analyse](../01_donnees/05_analyse_donnees.ipynb)  
**Suite :** `03_preparation_modele.ipynb`


## 1. Imports & chemins


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "patients.csv").is_file() and (p / "Sujet.md").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

from src.data.paths import ProjectPaths
from notebooks._utils.schema import FEATURES_CLES, SCHEMA_ATTENDU

paths = ProjectPaths(root=ROOT)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
print("Racine :", ROOT)
print("Curated :", paths.curated)
print("Models  :", paths.models)


## 2. Copie raw + chargement CSV → DataFrames


In [ ]:
from src.data.load import copy_sources_to_raw, load_csv

copy_sources_to_raw(paths)

dfs: dict[str, pd.DataFrame] = {}
for fichier in SCHEMA_ATTENDU:
    dfs[fichier] = load_csv(paths, fichier)
    print(f"✓ {fichier:25s} → {len(dfs[fichier]):>7,} × {dfs[fichier].shape[1]} cols")

print(f"\n{len(dfs)} DataFrames chargés.")


## 3. Vérification des colonnes attendues


In [ ]:
rapport = []
for fichier, attendues in SCHEMA_ATTENDU.items():
    df = dfs[fichier]
    cols, attendues_set = set(df.columns), set(attendues)
    manquantes = sorted(attendues_set - cols)
    en_trop = sorted(cols - attendues_set)
    rapport.append({
        "fichier": fichier,
        "lignes": len(df),
        "conforme": not manquantes and not en_trop,
        "manquantes": ", ".join(manquantes) or "—",
        "en_trop": ", ".join(en_trop) or "—",
    })

rapport_df = pd.DataFrame(rapport)
display(rapport_df)
nb_ko = (~rapport_df["conforme"]).sum()
if nb_ko:
    raise AssertionError(f"{nb_ko} fichier(s) non conforme(s).")
print("✓ Colonnes conformes.")


## 4. Pseudonymisation (vault vs patients pseudo)


In [ ]:
from src.data.identity import persist_identity_split, split_identity_and_pseudonymise

patients = dfs["patients.csv"]
vault, patients_pseudo = split_identity_and_pseudonymise(patients)
persist_identity_split(paths, vault, patients_pseudo)

print("Vault (nominatif) :", len(vault), "lignes →", paths.vault)
print("Patients pseudo   :", len(patients_pseudo), "lignes")
print("Colonnes vault    :", list(vault.columns))
print("Colonnes pseudo   :", list(patients_pseudo.columns))
# Ne pas afficher NomPrenom hors coffre
display(patients_pseudo.head(3))


## 5. Règles qualité (durée, biologie, objets connectés)


In [ ]:
from src.data.quality import (
    filter_objets_connectes_bons,
    harmonize_biologie,
    recalculate_duree_sejour,
)

sejours_brut = dfs["sejours.csv"]
sejours = recalculate_duree_sejour(sejours_brut)
biologies = harmonize_biologie(dfs["biologies.csv"])
objets = filter_objets_connectes_bons(dfs["objets_connectes.csv"])

delta_duree = (sejours["DureeSejour"] != sejours_brut["DureeSejour"]).sum()
print(f"Durées séjour recalculées (différentes) : {delta_duree}")
print(f"Biologies harmonisées : {len(biologies):,} lignes")
print(f"Objets connectés QualiteSignal=Bon : {len(objets):,} / {len(dfs['objets_connectes.csv']):,}")
display(sejours[["SejourID", "DureeSejour", "ModeSortie"]].head())


## 6. Cohorte domicile + split patient-level


In [ ]:
from src.data.cohort import filter_domicile, patient_level_split

sejours_dom = filter_domicile(sejours)
sejours_el = patient_level_split(sejours_dom)

print("Séjours totaux     :", len(sejours))
print("ModeSortie=Domicile:", len(sejours_dom))
print("Après split        :", len(sejours_el))
print(sejours_el["split"].value_counts().to_string())
print("Prévalence Readmission30j :", round(float(sejours_el["Readmission30j"].mean()), 4))
display(sejours_el.head())


## 7. Synthèse nettoyage


In [ ]:
synthese = pd.DataFrame([
    {"etape": "patients_pseudo", "n": len(patients_pseudo)},
    {"etape": "sejours_eligibles", "n": len(sejours_el)},
    {"etape": "biologies", "n": len(biologies)},
    {"etape": "objets_bons", "n": len(objets)},
])
display(synthese)
print("✓ Nettoyage terminé — enchaînez avec 03_preparation_modele.ipynb")
